In [ ]:
# Building a natural language processing (NLP) model for sentiment analysis or text classification.
from tensorflow.keras.datasets import imdb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Convert integer sequences to text using Keras's word index
def decode_review(sequence, word_index):
    reverse_word_index = {value: key for key, value in word_index.items()}
    # Words are offset by 3 in Keras IMDB (0,1,2 are reserved)
    return " ".join([reverse_word_index.get(i - 3, "?") for i in sequence if i >= 3])

if __name__ == "__main__":
    # Load IMDB dataset (num_words=10000 means only the top 10k words)
    (X_train_seq, y_train), (X_test_seq, y_test) = imdb.load_data(num_words=10)
    word_index = imdb.get_word_index()

    # Convert integer sequences to text
    X_train = [decode_review(seq, word_index) for seq in X_train_seq]
    X_test = [decode_review(seq, word_index) for seq in X_test_seq]

    # Build and train pipeline
    pipeline = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2),
        LogisticRegression(max_iter=1000)
    )

    print("Training model...")
    pipeline.fit(X_train, y_train)
    print("Training complete.")

    # Evaluate
    y_pred = pipeline.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

    # Save model
    joblib.dump(pipeline, "imdb_text_clf_model.joblib")
    print("Model saved to imdb_text_clf_model.joblib")

    # Test prediction
    example_review = "The movie was an absolute masterpiece with stunning visuals."
    pred = pipeline.predict([example_review])[0]
    print(f"Example review prediction: {'Positive' if pred == 1 else 'Negative'}")


Training model...
Training complete.
Accuracy: 0.58316
              precision    recall  f1-score   support

    Negative       0.58      0.59      0.58     12500
    Positive       0.58      0.58      0.58     12500

    accuracy                           0.58     25000
   macro avg       0.58      0.58      0.58     25000
weighted avg       0.58      0.58      0.58     25000

Model saved to imdb_text_clf_model.joblib
Example review prediction: Negative
